# Samatrix AIML Practice: House Price Prediction

This starter notebook is for the Samatrix House Price Prediction challenge.

Recommended workflow:

1. Work in Google Colab or local Jupyter.
2. Train your model in this notebook.
3. Generate `predictions.csv`.
4. Upload `predictions.csv` back to Samatrix for scoring.
5. Recommended: upload your completed `notebook.ipynb` as evidence of your work.

Samatrix scores the CSV. It does not execute this notebook.


## 1. Load the datasets

For now, download `train.csv`, `test.csv`, and `sample_submission.csv` from the Samatrix challenge page and upload them into Colab.

Later, once the platform is on a public HTTPS domain, this notebook can be updated to download the files automatically from Samatrix.


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('.')

train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'
sample_submission_path = DATA_DIR / 'sample_submission.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample submission shape:', sample_submission.shape)

train.head()


## 2. Inspect the data

Before training, look at the columns, missing values, and target distribution.


In [ ]:
print(train.info())
print('\nMissing values in train:')
print(train.isna().sum())

print('\nMissing values in test:')
print(test.isna().sum())


## 3. Build a simple baseline model

This baseline uses all columns except `id` and `price` as features.

You should improve this later with better feature engineering, validation, and model selection.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

ID_COLUMN = 'id'
TARGET_COLUMN = 'price'

feature_columns = [col for col in train.columns if col not in [ID_COLUMN, TARGET_COLUMN]]

X = train[feature_columns].copy()
y = train[TARGET_COLUMN].copy()
X_test = test[feature_columns].copy()

# Basic encoding for categorical columns, if any exist.
combined = pd.concat([X, X_test], axis=0)
combined = pd.get_dummies(combined, drop_first=False)

X_encoded = combined.iloc[:len(X)].copy()
X_test_encoded = combined.iloc[len(X):].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

model.fit(X_train, y_train)
valid_predictions = model.predict(X_valid)

rmse = mean_squared_error(y_valid, valid_predictions, squared=False)
print('Validation RMSE:', rmse)


## 4. Train final model and create predictions.csv

The final CSV must contain exactly these columns:

- `id`
- `prediction`


In [ ]:
final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

final_model.fit(X_encoded, y)
test_predictions = final_model.predict(X_test_encoded)

submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    'prediction': test_predictions,
})

submission.to_csv('predictions.csv', index=False)
submission.head()


## 5. Download and submit

Upload the required file to Samatrix:

1. `predictions.csv`

Recommended evidence:

2. Your completed notebook as `notebook.ipynb`

In Colab, download the notebook using:

`File > Download > Download .ipynb`


In [ ]:
# If you are using Google Colab, uncomment this to download predictions.csv.
# from google.colab import files
# files.download('predictions.csv')

print('Created predictions.csv. Upload it to Samatrix. Recommended: also upload your completed notebook.ipynb as evidence.')
